# YOLOv8 TensorRT Optimization & Quantization Benchmark

**Runtime**: Google Colab A100 (required for TensorRT)

This notebook benchmarks the trained YOLOv8-Large wafer defect model across multiple export formats:
- **PyTorch** (baseline)
- **ONNX** (current production format)
- **TensorRT FP32** (GPU-optimized)
- **TensorRT FP16** (half-precision, 2x faster)
- **TensorRT INT8** (4x faster, requires calibration)

## Expected Results
| Format | Latency (A100) | Size | Accuracy Impact |
|--------|----------|------|---------|
| PyTorch | ~8ms | 84MB | Baseline |
| ONNX | ~4ms | 167MB | None |
| TensorRT FP32 | ~3ms | ~170MB | None |
| TensorRT FP16 | ~1.5ms | ~85MB | Negligible (<0.1% mAP) |
| TensorRT INT8 | ~0.8ms | ~45MB | Minor (<0.5% mAP) |

In [ ]:
# Step 1: Verify GPU and install dependencies
!nvidia-smi
!pip install -q ultralytics>=8.1 onnx onnxruntime-gpu tensorrt

In [ ]:
# Step 2: Upload or download the trained model
import os
from pathlib import Path

# Option A: Upload from local (uncomment)
# from google.colab import files
# uploaded = files.upload()  # Upload best.pt

# Option B: Download from GitHub release or Google Drive
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

if not (MODEL_DIR / 'best.pt').exists():
    print('Please upload best.pt to models/ directory')
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        with open(MODEL_DIR / 'best.pt', 'wb') as f:
            f.write(data)
    print(f'Saved model: {(MODEL_DIR / "best.pt").stat().st_size / 1e6:.1f} MB')
else:
    print(f'Model found: {(MODEL_DIR / "best.pt").stat().st_size / 1e6:.1f} MB')

In [ ]:
# Step 3: Load model and verify
from ultralytics import YOLO
import torch

model = YOLO('models/best.pt')
print(f'Model: {model.model.names}')
print(f'Parameters: {sum(p.numel() for p in model.model.parameters()):,}')
print(f'Device: {next(model.model.parameters()).device}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Step 4: Export to all formats
import time

exports = {}

# ONNX
print('=== Exporting ONNX ===')
t0 = time.time()
onnx_path = model.export(format='onnx', imgsz=640, dynamic=True, simplify=True, opset=17)
exports['onnx'] = {'path': onnx_path, 'time': time.time() - t0}
print(f'ONNX export: {time.time() - t0:.1f}s')

# TensorRT FP32
print('\n=== Exporting TensorRT FP32 ===')
t0 = time.time()
trt_fp32_path = model.export(format='engine', imgsz=640, half=False, int8=False)
exports['trt_fp32'] = {'path': trt_fp32_path, 'time': time.time() - t0}
print(f'TensorRT FP32 export: {time.time() - t0:.1f}s')

# TensorRT FP16
print('\n=== Exporting TensorRT FP16 ===')
t0 = time.time()
trt_fp16_path = model.export(format='engine', imgsz=640, half=True, int8=False)
exports['trt_fp16'] = {'path': trt_fp16_path, 'time': time.time() - t0}
print(f'TensorRT FP16 export: {time.time() - t0:.1f}s')

# TensorRT INT8 (needs calibration data)
print('\n=== Exporting TensorRT INT8 ===')
t0 = time.time()
try:
    trt_int8_path = model.export(format='engine', imgsz=640, half=False, int8=True, data='data.yaml')
    exports['trt_int8'] = {'path': trt_int8_path, 'time': time.time() - t0}
    print(f'TensorRT INT8 export: {time.time() - t0:.1f}s')
except Exception as e:
    print(f'INT8 export failed (needs calibration data): {e}')
    exports['trt_int8'] = None

print('\n=== Export Summary ===')
for name, info in exports.items():
    if info:
        size_mb = Path(info['path']).stat().st_size / 1e6
        print(f'{name:12s}: {size_mb:7.1f} MB  (export time: {info["time"]:.1f}s)')

In [ ]:
# Step 5: Benchmark all formats
import numpy as np
import json

def benchmark_model(model_path, n_warmup=50, n_runs=200, imgsz=640):
    """Benchmark inference latency."""
    m = YOLO(model_path)
    dummy = np.random.randint(0, 255, (imgsz, imgsz, 3), dtype=np.uint8)

    # Warmup
    for _ in range(n_warmup):
        m(dummy, verbose=False)

    # Timed runs
    latencies = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        m(dummy, verbose=False)
        latencies.append((time.perf_counter() - t0) * 1000)

    arr = np.array(latencies)
    return {
        'mean_ms': round(float(arr.mean()), 3),
        'median_ms': round(float(np.median(arr)), 3),
        'p95_ms': round(float(np.percentile(arr, 95)), 3),
        'p99_ms': round(float(np.percentile(arr, 99)), 3),
        'min_ms': round(float(arr.min()), 3),
        'max_ms': round(float(arr.max()), 3),
        'fps': round(float(1000 / arr.mean()), 1),
        'std_ms': round(float(arr.std()), 3),
    }

results = {}

# PyTorch baseline
print('Benchmarking PyTorch...')
results['pytorch'] = benchmark_model('models/best.pt')
results['pytorch']['size_mb'] = round(Path('models/best.pt').stat().st_size / 1e6, 1)
print(f"  Mean: {results['pytorch']['mean_ms']:.2f}ms | FPS: {results['pytorch']['fps']}")

# ONNX
if exports.get('onnx'):
    print('Benchmarking ONNX...')
    results['onnx'] = benchmark_model(exports['onnx']['path'])
    results['onnx']['size_mb'] = round(Path(exports['onnx']['path']).stat().st_size / 1e6, 1)
    print(f"  Mean: {results['onnx']['mean_ms']:.2f}ms | FPS: {results['onnx']['fps']}")

# TensorRT FP32
if exports.get('trt_fp32'):
    print('Benchmarking TensorRT FP32...')
    results['trt_fp32'] = benchmark_model(exports['trt_fp32']['path'])
    results['trt_fp32']['size_mb'] = round(Path(exports['trt_fp32']['path']).stat().st_size / 1e6, 1)
    print(f"  Mean: {results['trt_fp32']['mean_ms']:.2f}ms | FPS: {results['trt_fp32']['fps']}")

# TensorRT FP16
if exports.get('trt_fp16'):
    print('Benchmarking TensorRT FP16...')
    results['trt_fp16'] = benchmark_model(exports['trt_fp16']['path'])
    results['trt_fp16']['size_mb'] = round(Path(exports['trt_fp16']['path']).stat().st_size / 1e6, 1)
    print(f"  Mean: {results['trt_fp16']['mean_ms']:.2f}ms | FPS: {results['trt_fp16']['fps']}")

# TensorRT INT8
if exports.get('trt_int8'):
    print('Benchmarking TensorRT INT8...')
    results['trt_int8'] = benchmark_model(exports['trt_int8']['path'])
    results['trt_int8']['size_mb'] = round(Path(exports['trt_int8']['path']).stat().st_size / 1e6, 1)
    print(f"  Mean: {results['trt_int8']['mean_ms']:.2f}ms | FPS: {results['trt_int8']['fps']}")

# Save results
with open('tensorrt_benchmark_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nResults saved to tensorrt_benchmark_results.json')

In [ ]:
# Step 6: Accuracy validation (mAP comparison across formats)
# Validate that quantized models maintain accuracy

print('=== Accuracy Validation ===')
print('Running validation on test set for each format...\n')

accuracy_results = {}

for name, info in [('pytorch', {'path': 'models/best.pt'})] + list(exports.items()):
    if info is None:
        continue
    try:
        m = YOLO(info['path'])
        metrics = m.val(data='data.yaml', imgsz=640, verbose=False)
        accuracy_results[name] = {
            'mAP50': round(float(metrics.box.map50), 4),
            'mAP50_95': round(float(metrics.box.map), 4),
            'precision': round(float(metrics.box.mp), 4),
            'recall': round(float(metrics.box.mr), 4),
        }
        print(f'{name:12s}: mAP@50={accuracy_results[name]["mAP50"]:.4f}  mAP@50:95={accuracy_results[name]["mAP50_95"]:.4f}')
    except Exception as e:
        print(f'{name:12s}: validation failed — {e}')

# Save
with open('tensorrt_accuracy_results.json', 'w') as f:
    json.dump(accuracy_results, f, indent=2)

In [ ]:
# Step 7: Publication-quality comparison chart
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 12

formats = list(results.keys())
latencies = [results[f]['mean_ms'] for f in formats]
fps_vals = [results[f]['fps'] for f in formats]
sizes = [results[f]['size_mb'] for f in formats]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Latency comparison
colors = ['#3b82f6', '#22c55e', '#f59e0b', '#ef4444', '#8b5cf6']
bars = axes[0].bar(formats, latencies, color=colors[:len(formats)])
axes[0].set_title('Inference Latency (lower is better)', fontweight='bold')
axes[0].set_ylabel('Latency (ms)')
for bar, val in zip(bars, latencies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.2f}ms', ha='center', fontweight='bold')

# FPS comparison
bars = axes[1].bar(formats, fps_vals, color=colors[:len(formats)])
axes[1].set_title('Throughput (higher is better)', fontweight='bold')
axes[1].set_ylabel('FPS')
for bar, val in zip(bars, fps_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{val:.0f}', ha='center', fontweight='bold')

# Model size comparison
bars = axes[2].bar(formats, sizes, color=colors[:len(formats)])
axes[2].set_title('Model Size (lower is better)', fontweight='bold')
axes[2].set_ylabel('Size (MB)')
for bar, val in zip(bars, sizes):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{val:.0f}MB', ha='center', fontweight='bold')

plt.suptitle('YOLOv8-Large Wafer Defect Detection — TensorRT Optimization Benchmark\nGoogle Colab A100',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('tensorrt_benchmark_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to tensorrt_benchmark_chart.png')

In [ ]:
# Step 8: Summary table
import pandas as pd

summary_data = []
for fmt in results:
    row = {
        'Format': fmt.upper().replace('_', ' '),
        'Latency (ms)': results[fmt]['mean_ms'],
        'P95 (ms)': results[fmt]['p95_ms'],
        'FPS': results[fmt]['fps'],
        'Size (MB)': results[fmt]['size_mb'],
    }
    if fmt in accuracy_results:
        row['mAP@50'] = accuracy_results[fmt]['mAP50']
        row['mAP@50:95'] = accuracy_results[fmt]['mAP50_95']
    summary_data.append(row)

df = pd.DataFrame(summary_data)
print('\n=== Final Benchmark Summary ===')
print(df.to_string(index=False))
df.to_csv('tensorrt_benchmark_summary.csv', index=False)
print('\nSaved to tensorrt_benchmark_summary.csv')

In [ ]:
# Step 9: Download results
from google.colab import files
import shutil

# Create results bundle
os.makedirs('tensorrt_results', exist_ok=True)
for f in ['tensorrt_benchmark_results.json', 'tensorrt_accuracy_results.json',
          'tensorrt_benchmark_chart.png', 'tensorrt_benchmark_summary.csv']:
    if os.path.exists(f):
        shutil.copy(f, 'tensorrt_results/')

# Also copy the FP16 engine (best balance of speed/accuracy)
if exports.get('trt_fp16'):
    fp16_path = exports['trt_fp16']['path']
    shutil.copy(fp16_path, 'tensorrt_results/best_fp16.engine')
    print(f'FP16 engine: {Path(fp16_path).stat().st_size / 1e6:.1f} MB')

shutil.make_archive('tensorrt_results', 'zip', 'tensorrt_results')
files.download('tensorrt_results.zip')
print('\nDownload started: tensorrt_results.zip')

## Next Steps

1. Copy `best_fp16.engine` to `triton_model_repo/yolo_wafer_defect/2/model.plan`
2. Update Triton `config.pbtxt` to serve TensorRT engine
3. Copy benchmark chart to `outputs/tensorrt_benchmark.png`
4. Update README with TensorRT results